# Comparacion de Modelos

Resumen comparativo de los 4 modelos entrenados.

**Importante:** Arbol de Decision y SVM predicen `is_hit` (exito comercial binario).
MLP predice `imdb_clase` (calidad percibida en 3 clases). Son problemas complementarios, no directamente comparables.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from sklearn.metrics import roc_auc_score, RocCurveDisplay, f1_score, accuracy_score


## 1. Cargar datos y modelos

In [ ]:
X_test_clf = pd.read_csv('../data/processed/X_test_clf.csv')
y_test_clf = pd.read_csv('../data/processed/y_test_clf.csv').squeeze()
X_test_nn  = pd.read_csv('../data/processed/X_test_nn.csv')
y_test_nn  = pd.read_csv('../data/processed/y_test_nn.csv').squeeze()

dt  = joblib.load('../models/decision_tree.joblib')
svm = joblib.load('../models/svm.joblib')
mlp = joblib.load('../models/mlp.joblib')
print('Modelos cargados')


## 2. Calcular metricas

In [ ]:
# DT y SVM: target is_hit (binario)
y_pred_dt  = dt.predict(X_test_clf)
y_proba_dt = dt.predict_proba(X_test_clf)[:, 1]
y_pred_svm  = svm.predict(X_test_clf)
y_proba_svm = svm.predict_proba(X_test_clf)[:, 1]

# MLP: target imdb_clase (multiclase)
y_pred_mlp = mlp.predict(X_test_nn)

tabla = pd.DataFrame([
    {'Modelo': 'Arbol de Decision', 'Target': 'is_hit',
     'Accuracy': round(float(accuracy_score(y_test_clf, y_pred_dt)), 4),
     'F1-weighted': round(float(f1_score(y_test_clf, y_pred_dt, average='weighted')), 4),
     'ROC-AUC': round(float(roc_auc_score(y_test_clf, y_proba_dt)), 4)},
    {'Modelo': 'SVM', 'Target': 'is_hit',
     'Accuracy': round(float(accuracy_score(y_test_clf, y_pred_svm)), 4),
     'F1-weighted': round(float(f1_score(y_test_clf, y_pred_svm, average='weighted')), 4),
     'ROC-AUC': round(float(roc_auc_score(y_test_clf, y_proba_svm)), 4)},
    {'Modelo': 'MLP', 'Target': 'imdb_clase',
     'Accuracy': round(float(accuracy_score(y_test_nn, y_pred_mlp)), 4),
     'F1-weighted': round(float(f1_score(y_test_nn, y_pred_mlp, average='weighted')), 4),
     'ROC-AUC': '-  (multiclase, ver por clase)'},
])
print(tabla.to_string(index=False))


## 3. Curvas ROC superpuestas (DT vs SVM)

Comparable porque ambos predicen `is_hit` en el mismo test set.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(
    y_test_clf, y_proba_dt,
    name=f'Arbol Decision (AUC={roc_auc_score(y_test_clf, y_proba_dt):.3f})', ax=ax
)
RocCurveDisplay.from_predictions(
    y_test_clf, y_proba_svm,
    name=f'SVM (AUC={roc_auc_score(y_test_clf, y_proba_svm):.3f})', ax=ax
)
ax.plot([0, 1], [0, 1], 'k--', label='Aleatorio')
ax.set_title('Curvas ROC - Arbol de Decision vs SVM (target: is_hit)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../reports/figures/21_roc_comparacion.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. MLP: F1 por clase (Bajo / Medio / Alto)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test_nn, y_pred_mlp, target_names=['Bajo', 'Medio', 'Alto']))


## 5. Guardar metricas consolidadas

In [ ]:
metricas = {
    'decision_tree': {
        'modelo': 'Arbol de Decision', 'target': 'is_hit',
        'accuracy': round(float(accuracy_score(y_test_clf, y_pred_dt)), 4),
        'f1_weighted': round(float(f1_score(y_test_clf, y_pred_dt, average='weighted')), 4),
        'roc_auc': round(float(roc_auc_score(y_test_clf, y_proba_dt)), 4),
    },
    'svm': {
        'modelo': 'SVM', 'target': 'is_hit',
        'accuracy': round(float(accuracy_score(y_test_clf, y_pred_svm)), 4),
        'f1_weighted': round(float(f1_score(y_test_clf, y_pred_svm, average='weighted')), 4),
        'roc_auc': round(float(roc_auc_score(y_test_clf, y_proba_svm)), 4),
    },
    'mlp': {
        'modelo': 'Red Neuronal (MLP)', 'target': 'imdb_clase',
        'accuracy': round(float(accuracy_score(y_test_nn, y_pred_mlp)), 4),
        'f1_macro': round(float(f1_score(y_test_nn, y_pred_mlp, average='macro')), 4),
        'f1_weighted': round(float(f1_score(y_test_nn, y_pred_mlp, average='weighted')), 4),
    },
}

with open('../models/metricas.json', 'w', encoding='utf-8') as f:
    json.dump(metricas, f, indent=2, ensure_ascii=False)
print('Guardado: models/metricas.json')
